# Asymptotic Safety – Research-Style Exploratory Notebook

This notebook is a more advanced numerical companion to a theoretical essay on asymptotically safe quantum gravity.

It includes:

- automatic fixed-point search for toy beta-function systems
- stability matrix and critical exponent analysis
- 2D and 3D RG flow visualization
- approximation to UV critical surfaces
- RG-improved cosmology toy models
- toy cosmic-string stochastic background illustration

The systems here are **illustrative toy models**, not full functional-renormalization-group computations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import root
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import sympy as sp

plt.rcParams["figure.figsize"] = (8, 6)

## 1. Toy beta-function system

We work with a three-coupling toy model:
- `g`: dimensionless Newton coupling
- `lam`: dimensionless cosmological constant
- `h`: Horndeski-like braiding coupling

The chosen beta functions are designed to illustrate:
- an interacting fixed point for gravity,
- suppression of the braiding coupling,
- nontrivial flow in coupling space.

In [ ]:
def beta_vec(y):
    g, lam, h = y
    dg = 2*g - g**2
    dlam = -2*lam + g - 0.2*h**2
    dh = -h*(1 + g) + 0.1*lam*h
    return np.array([dg, dlam, dh], dtype=float)

def beta_ode(t, y):
    return beta_vec(y)

## 2. Automatic fixed-point search

We search for roots of the beta functions from multiple initial guesses.

In [ ]:
guesses = [
    [0.0, 0.0, 0.0],
    [0.5, 0.1, 0.1],
    [1.0, 0.5, 0.0],
    [2.0, 1.0, 0.0],
    [2.0, 1.0, 0.5],
    [2.0, 1.0, -0.5],
    [1.5, 0.7, 0.2],
]

fixed_points = []
for guess in guesses:
    sol = root(lambda x: beta_vec(x), guess)
    if sol.success:
        fp = np.round(sol.x, 10)
        if not any(np.allclose(fp, old, atol=1e-7) for old in fixed_points):
            fixed_points.append(fp)

fixed_points

## 3. Stability matrix and critical exponents

We compute the Jacobian of the beta functions at each fixed point.

In [ ]:
g_s, lam_s, h_s = sp.symbols('g lam h')

beta_sym = sp.Matrix([
    2*g_s - g_s**2,
    -2*lam_s + g_s - sp.Rational(1,5)*h_s**2,
    -h_s*(1 + g_s) + sp.Rational(1,10)*lam_s*h_s
])

J_sym = beta_sym.jacobian([g_s, lam_s, h_s])
J_sym

In [ ]:
def jacobian_at(fp):
    subs = {g_s: fp[0], lam_s: fp[1], h_s: fp[2]}
    J = np.array(J_sym.subs(subs), dtype=float)
    return J

for i, fp in enumerate(fixed_points, start=1):
    J = jacobian_at(fp)
    eigvals, eigvecs = np.linalg.eig(J)
    print(f"Fixed point {i}: {fp}")
    print("Jacobian:")
    print(J)
    print("Eigenvalues:")
    print(eigvals)
    print("-"*60)

## 4. 2D RG flow in the gravitational sector

We first visualize the flow projected onto `(g, λ)` with `h = 0`.

In [ ]:
g_vals = np.linspace(-0.2, 3.0, 25)
lam_vals = np.linspace(-0.5, 1.8, 25)
G, L = np.meshgrid(g_vals, lam_vals)

DG = 2*G - G**2
DL = -2*L + G

plt.streamplot(G, L, DG, DL, density=1.2)
plt.scatter([fp[0] for fp in fixed_points], [fp[1] for fp in fixed_points], s=40)
plt.xlabel("g")
plt.ylabel("lambda")
plt.title("Projected RG flow in the (g, lambda) plane")
plt.show()

## 5. 3D RG flow trajectories

We integrate several trajectories in full `(g, λ, h)` space.

In [ ]:
initial_conditions = [
    [0.2, 0.0, 0.8],
    [0.5, 0.4, -0.7],
    [1.2, 0.2, 0.5],
    [2.5, 1.5, 0.3],
    [1.8, 1.0, -0.3]
]

fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection='3d')

for y0 in initial_conditions:
    sol = solve_ivp(beta_ode, [0, -8], y0, dense_output=True, max_step=0.05)
    t = np.linspace(0, -8, 400)
    y = sol.sol(t)
    ax.plot(y[0], y[1], y[2])

for fp in fixed_points:
    ax.scatter(fp[0], fp[1], fp[2], s=50)

ax.set_xlabel("g")
ax.set_ylabel("lambda")
ax.set_zlabel("h")
ax.set_title("3D RG trajectories")
plt.show()

## 6. Approximate UV critical surface near an interacting fixed point

We linearize around the interacting fixed point and identify eigenvectors with positive real part in the convention
`k ∂_k δg = B δg`.

In [ ]:
# choose the interacting fixed point with largest g
fp_target = max(fixed_points, key=lambda x: x[0])
J = jacobian_at(fp_target)
eigvals, eigvecs = np.linalg.eig(J)

print("Target fixed point:", fp_target)
print("Eigenvalues:", eigvals)

# Build a small local patch from eigenvectors whose real part is positive
relevant = [i for i, ev in enumerate(eigvals) if np.real(ev) > 0]
irrelevant = [i for i, ev in enumerate(eigvals) if np.real(ev) <= 0]
print("Relevant indices:", relevant)
print("Irrelevant indices:", irrelevant)

In [ ]:
# Visualize a local patch if there are at least two relevant directions; otherwise use the first two eigendirections
dirs = relevant[:2] if len(relevant) >= 2 else [0, 1]
v1 = np.real(eigvecs[:, dirs[0]])
v2 = np.real(eigvecs[:, dirs[1]])

u = np.linspace(-0.3, 0.3, 15)
v = np.linspace(-0.3, 0.3, 15)
U, V = np.meshgrid(u, v)

X = fp_target[0] + U*v1[0] + V*v2[0]
Y = fp_target[1] + U*v1[1] + V*v2[1]
Z = fp_target[2] + U*v1[2] + V*v2[2]

fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, alpha=0.6)
ax.scatter(fp_target[0], fp_target[1], fp_target[2], s=70)
ax.set_xlabel("g")
ax.set_ylabel("lambda")
ax.set_zlabel("h")
ax.set_title("Local approximation to UV critical surface")
plt.show()

## 7. Weak-gravity bound illustration

We now add a toy matter coupling `y` with beta function

\[
\beta_y = y(1-y-g).
\]

The interacting matter fixed point is at `y* = 1 - g`, so it disappears for `g > 1`.

In [ ]:
g_range = np.linspace(0, 2.0, 300)
y_star = 1 - g_range

plt.plot(g_range, y_star, label=r"$y_* = 1-g$")
plt.axhline(0, linewidth=1)
plt.axvline(1, linestyle="--", label="critical gravity")
plt.xlabel("g")
plt.ylabel("matter fixed point")
plt.title("Toy weak-gravity bound")
plt.legend()
plt.show()

## 8. RG-improved Newton constant

A simple toy model for RG improvement is

\[
G(k)=\frac{G_0}{1+G_0 k^2}.
\]

This illustrates weakening of gravity at high scales in a dimensionful parametrization.

In [ ]:
k = np.logspace(-4, 3, 500)
G0 = 1.0
Gk = G0 / (1 + G0 * k**2)

plt.loglog(k, Gk)
plt.xlabel("scale k")
plt.ylabel("G(k)")
plt.title("Toy RG-improved Newton constant")
plt.show()

## 9. RG-improved cosmology toy model

We use a simple modified Friedmann equation

\[
H^2(a) = \frac{8\pi G(H)}{3} \rho(a) + \frac{\Lambda(H)}{3},
\]

but for illustration we replace the running scale by a function of the scale factor.
This is not a precision model; it simply shows how running couplings can alter background expansion.

In [ ]:
a = np.logspace(-3, 0, 400)

# toy matter density
rho_m0 = 1.0
rho_a = rho_m0 / a**3

# toy RG scale identified with inverse scale factor
k_a = 1/a - 1
k_a = np.maximum(k_a, 0)

G_run = 1/(1 + 0.3*k_a**2)
Lambda_run = 0.7 + 0.05*np.exp(-k_a)

H2 = (8*np.pi*G_run/3.0)*rho_a + Lambda_run/3.0
H = np.sqrt(H2)

plt.loglog(a, H, label="RG-improved toy H(a)")
plt.xlabel("scale factor a")
plt.ylabel("H(a)")
plt.title("Toy RG-improved cosmological background")
plt.legend()
plt.show()

## 10. Effective equation-of-state proxy

We define a simple proxy

\[
w_{\rm eff}(a) = -1 - \frac{1}{3}\frac{d\ln H^2}{d\ln a},
\]

to see how running couplings can perturb a pure-Λ behavior.

In [ ]:
lnH2 = np.log(H2)
lna = np.log(a)
dlnH2_dlna = np.gradient(lnH2, lna)
w_eff = -1 - (1/3.0)*dlnH2_dlna

plt.semilogx(a, w_eff)
plt.xlabel("scale factor a")
plt.ylabel("w_eff(a)")
plt.title("Toy effective equation-of-state proxy")
plt.show()

## 11. Toy cosmic-string stochastic background

This is only an illustrative mapping from string tension to signal amplitude.

We take a toy spectrum
\[
\Omega_{\rm GW}(f) \propto (G\mu)^2 \left(\frac{f}{f_*}\right)^n
\]
with different string tensions to illustrate how changing the symmetry-breaking scale changes the signal.

In [ ]:
f = np.logspace(-10, -6, 400)
fstar = 1e-8
n = 0.3

tensions = [1e-11, 1e-10, 1e-9]
for Gmu in tensions:
    Omega = (Gmu**2) * (f/fstar)**n
    plt.loglog(f, Omega, label=fr"$G\mu={Gmu}$")

plt.xlabel("frequency [arb.]")
plt.ylabel(r"$\Omega_{m GW}(f)$ [arb.]")
plt.title("Toy cosmic-string stochastic background")
plt.legend()
plt.show()

## 12. Automatic phase portrait experiment

You can edit the beta functions and rerun the notebook to test:
- whether an interacting fixed point still exists,
- how many relevant directions survive,
- whether the weak-gravity threshold shifts,
- how RG-improved cosmology changes.

In [ ]:
def analyze_system(beta_function, guesses):
    fps = []
    for guess in guesses:
        sol = root(lambda x: beta_function(x), guess)
        if sol.success:
            fp = np.round(sol.x, 10)
            if not any(np.allclose(fp, old, atol=1e-7) for old in fps):
                fps.append(fp)
    return fps

analyze_system(beta_vec, guesses)